# Titanic Experiment Sandbox

A compact notebook for quick experiments. The final assignment notebook stays unchanged; use this file to test feature/model ideas safely.

## 1. Setup

Change the values in `EXPERIMENT_CONFIG` to try small variations.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

In [ ]:
EXPERIMENT_CONFIG = {
    # Feature switches - 8 features with HasCabin + Embarked_C
    'use_fare_log': True,
    'use_has_cabin': True,
    'use_family_size': True,
    'use_embarked_c': True,
    'use_is_crew': False,
    'use_age_missing': False,
    'use_fare_per_person': False,

    # Age imputation
    'age_imputation': 'title',

    # Validation
    'cv_splits': 5,
    'random_state': 42,
}

EXPERIMENT_CONFIG

## 2. Load Data

In [ ]:
train = pd.read_csv('Datasets/train.csv')
test = pd.read_csv('Datasets/test.csv')

n_train = len(train)
y = train['Survived'].copy()
combined = pd.concat([train.drop(columns='Survived'), test], ignore_index=True)

print(train.shape, test.shape)
combined.head()

## 2.1 Quick EDA Visuals

These plots show the main survival signals before feature engineering.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.countplot(data=train, x='Survived', ax=axes[0, 0])
axes[0, 0].set_title('Target Distribution')

sns.barplot(data=train, x='Sex', y='Survived', ax=axes[0, 1])
axes[0, 1].set_title('Survival Rate by Sex')

sns.barplot(data=train, x='Pclass', y='Survived', ax=axes[1, 0])
axes[1, 0].set_title('Survival Rate by Passenger Class')

sns.histplot(data=train, x='Age', hue='Survived', bins=30, kde=True, ax=axes[1, 1])
axes[1, 1].set_title('Age Distribution by Survival')

plt.tight_layout()
plt.show()

In [ ]:
missing = train.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]

plt.figure(figsize=(8, 4))
sns.barplot(x=missing.values, y=missing.index, color='#4c78a8')
plt.title('Missing Values in Training Data')
plt.xlabel('Missing count')
plt.ylabel('Column')
plt.tight_layout()
plt.show()

## 3. Feature Engineering

This function keeps the whole feature pipeline in one place, so it is easy to modify.

In [ ]:
def build_features(raw_combined, n_train, config):
    df = raw_combined.copy()

    # Extract useful title information from names.
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    title_map = {
        'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
        'Lady': 'Rare', 'Countess': 'Rare', 'Capt': 'Rare', 'Col': 'Rare',
        'Don': 'Rare', 'Dr': 'Rare', 'Major': 'Rare', 'Rev': 'Rare',
        'Sir': 'Rare', 'Jonkheer': 'Rare', 'Dona': 'Rare'
    }
    df['Title'] = df['Title'].replace(title_map)
    df['Title'] = df['Title'].where(df['Title'].isin(['Mr', 'Mrs', 'Miss', 'Master']), 'Rare')

    # Optional compact family and cabin features.
    if config['use_family_size']:
        df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

    if config['use_has_cabin']:
        df['HasCabin'] = df['Cabin'].notna().astype(int)

    # EXPERIMENT: IsCrew = Fare=0 OR known crew ticket series
    if config.get('use_is_crew', False):
        crew_tickets = ['LINE', '239853', '239854', '239855', '239856', '239852']
        df['IsCrew'] = ((df['Fare'] == 0) | df['Ticket'].isin(crew_tickets)).astype(int)

    # EXPERIMENT: AgeMissing indicator (before imputation)
    if config.get('use_age_missing', False):
        df['AgeMissing'] = df['Age'].isna().astype(int)

    # Age imputation: title-only is simple; title + class is a common experiment.
    if config['age_imputation'] == 'title_pclass':
        age_medians = df.iloc[:n_train].groupby(['Title', 'Pclass'])['Age'].median()
        for (title, pclass), median_age in age_medians.items():
            mask = df['Age'].isna() & (df['Title'] == title) & (df['Pclass'] == pclass)
            df.loc[mask, 'Age'] = median_age
    else:
        age_medians = df.iloc[:n_train].groupby('Title')['Age'].median()
        for title, median_age in age_medians.items():
            df.loc[df['Age'].isna() & (df['Title'] == title), 'Age'] = median_age

    # Safety fallback in case a rare group still has no age median.
    df['Age'] = df['Age'].fillna(df.iloc[:n_train]['Age'].median())

    # Fare imputation by class, then optional log transform.
    fare_medians = df.iloc[:n_train].groupby('Pclass')['Fare'].median()
    for pclass, median_fare in fare_medians.items():
        df.loc[df['Fare'].isna() & (df['Pclass'] == pclass), 'Fare'] = median_fare

    if config['use_fare_log']:
        df['FareLog'] = np.log1p(df['Fare'])

    # EXPERIMENT: FarePerPerson = Fare / FamilySize (after Fare imputation)
    if config.get('use_fare_per_person', False):
        df['FarePerPerson'] = df['Fare'] / df['FamilySize'].clip(lower=1)

    # Encode categorical values into numeric model inputs.
    df['Pclass'] = df['Pclass'].map({1: 2, 2: 1, 3: 0})
    title_encoder = OrdinalEncoder(
        categories=[['Mr', 'Rare', 'Mrs', 'Miss', 'Master']],
        handle_unknown='use_encoded_value',
        unknown_value=-1
    )
    title_encoder.fit(df.iloc[:n_train][['Title']])
    df[['Title']] = title_encoder.transform(df[['Title']])
    df['Sex'] = df['Sex'].map({'female': 0, 'male': 1})

    if config['use_embarked_c']:
        embarked_mode = df.iloc[:n_train]['Embarked'].mode()[0]
        df['Embarked'] = df['Embarked'].fillna(embarked_mode)
        df['Embarked_C'] = (df['Embarked'] == 'C').astype(int)

    # Keep only compact, experiment-friendly features.
    features = ['Pclass', 'Title', 'Age', 'Sex']
    if config.get('use_fare_per_person', False):
        features.append('FarePerPerson')
    elif config['use_fare_log']:
        features.append('FareLog')
    else:
        features.append('Fare')
    if config['use_family_size']:
        features.append('FamilySize')
    if config['use_has_cabin']:
        features.append('HasCabin')
    if config.get('use_age_missing', False):
        features.append('AgeMissing')
    if config.get('use_is_crew', False):
        features.append('IsCrew')
    if config['use_embarked_c']:
        features.append('Embarked_C')

    X_all = df[features].copy()

    # Scaling helps Logistic Regression and does not hurt tree models much here.
    scale_cols = [col for col in ['Age', 'FareLog', 'Fare', 'FamilySize', 'FarePerPerson'] if col in X_all.columns]
    scaler = StandardScaler()
    scaler.fit(X_all.iloc[:n_train][scale_cols])
    X_all[scale_cols] = scaler.transform(X_all[scale_cols])

    return X_all.iloc[:n_train].copy(), X_all.iloc[n_train:].copy(), features

In [ ]:
X, X_test, feature_names = build_features(combined, n_train, EXPERIMENT_CONFIG)

print(X.shape, X_test.shape)
print(feature_names)
X.head()

## 3.1 Feature Visuals

Use these plots to see how the current experiment configuration changed the feature table.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.heatmap(X.corr(numeric_only=True), cmap='coolwarm', center=0, ax=axes[0])
axes[0].set_title('Feature Correlation Matrix')

feature_means = X.mean().sort_values()
sns.barplot(x=feature_means.values, y=feature_means.index, ax=axes[1], color='#72b7b2')
axes[1].set_title('Scaled Feature Means')
axes[1].set_xlabel('Mean value')

plt.tight_layout()
plt.show()

In [ ]:
plot_cols = [col for col in ['Age', 'FareLog', 'Fare', 'FamilySize'] if col in X.columns]

if plot_cols:
    X[plot_cols].hist(figsize=(12, 4), bins=25)
    plt.suptitle('Scaled Continuous Feature Distributions')
    plt.tight_layout()
    plt.show()

## 4. Baseline

This baseline uses the original unscaled age values.

In [ ]:
baseline_frame = combined.iloc[:n_train].copy()
baseline_frame['Title'] = baseline_frame['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
age_medians = baseline_frame.groupby('Title')['Age'].median()
for title, median_age in age_medians.items():
    baseline_frame.loc[baseline_frame['Age'].isna() & (baseline_frame['Title'] == title), 'Age'] = median_age
baseline_frame['Age'] = baseline_frame['Age'].fillna(baseline_frame['Age'].median())

baseline_pred = ((baseline_frame['Sex'] == 'female') | (baseline_frame['Age'] < 16)).astype(int)
baseline_metrics = {
    'Model': 'Baseline',
    'Accuracy': accuracy_score(y, baseline_pred),
    'Precision': precision_score(y, baseline_pred, zero_division=0),
    'Recall': recall_score(y, baseline_pred, zero_division=0),
    'F1': f1_score(y, baseline_pred, zero_division=0),
}
baseline_metrics

## 5. Models

Edit `MODEL_GRIDS` to add/remove models or change hyperparameters.

In [ ]:
MODEL_GRIDS = {
    'Random Forest': (
        RandomForestClassifier(random_state=42),
        {
            'n_estimators': [50, 100, 150, 200, 250, 300],
            'max_depth': [2, 3, 4, 5, 6, 7, 8, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2', None]
        }
    ),
    'Gradient Boosting': (
        GradientBoostingClassifier(random_state=42),
        {'n_estimators': [100, 150, 200, 250], 'learning_rate': [0.03, 0.05, 0.075, 0.1, 0.15], 'max_depth': [2, 3, 4, 5]}
    ),
}

In [ ]:
def evaluate_model(name, estimator, grid, X, y, cv):
    search = GridSearchCV(estimator, grid, cv=cv, scoring='f1', n_jobs=-1)
    search.fit(X, y)

    scores = cross_validate(search.best_estimator_, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    metrics = {
        'Model': name,
        'Accuracy': scores['test_accuracy'].mean(),
        'Precision': scores['test_precision'].mean(),
        'Recall': scores['test_recall'].mean(),
        'F1': scores['test_f1'].mean(),
    }
    return metrics, search.best_estimator_, search.best_params_

In [ ]:
cv = StratifiedKFold(
    n_splits=EXPERIMENT_CONFIG['cv_splits'],
    shuffle=True,
    random_state=EXPERIMENT_CONFIG['random_state']
)
scoring = {'accuracy': 'accuracy', 'precision': 'precision', 'recall': 'recall', 'f1': 'f1'}

results = [baseline_metrics]
best_estimators = {}
best_params = {}

for name, (estimator, grid) in MODEL_GRIDS.items():
    metrics, best_estimator, params = evaluate_model(name, estimator, grid, X, y, cv)
    results.append(metrics)
    best_estimators[name] = best_estimator
    best_params[name] = params

pd.DataFrame(best_params).T

## 6. Optional Voting Ensemble

This uses the tuned models from the previous step. Comment this cell out if you want a faster run.

In [ ]:
voting = VotingClassifier(
    estimators=[
        ('rf', best_estimators['Random Forest']),
        ('gb', best_estimators['Gradient Boosting']),
    ],
    voting='soft'
)

voting_scores = cross_validate(voting, X, y, cv=cv, scoring=scoring, n_jobs=-1)
voting_metrics = {
    'Model': 'Voting Ensemble',
    'Accuracy': voting_scores['test_accuracy'].mean(),
    'Precision': voting_scores['test_precision'].mean(),
    'Recall': voting_scores['test_recall'].mean(),
    'F1': voting_scores['test_f1'].mean(),
}
results.append(voting_metrics)
best_estimators['Voting Ensemble'] = voting
voting_metrics

## 7. Compare Results

In [ ]:
results_df = pd.DataFrame(results).set_index('Model')
for metric in ['Accuracy', 'Precision', 'Recall', 'F1']:
    results_df[f'{metric} Δ vs Baseline'] = results_df[metric] - results_df.loc['Baseline', metric]

results_df.round(3).sort_values('F1', ascending=False)

In [ ]:
plot_results = results_df[['Accuracy', 'Precision', 'Recall', 'F1']].sort_values('F1')

ax = plot_results.plot(kind='barh', figsize=(10, 6))
ax.set_title('Model Comparison')
ax.set_xlabel('Cross-validated score')
ax.set_xlim(0, 1)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

delta_results = results_df[['Accuracy Δ vs Baseline', 'F1 Δ vs Baseline']].sort_values('F1 Δ vs Baseline')
ax = delta_results.plot(kind='barh', figsize=(10, 4), color=['#72b7b2', '#f58518'])
ax.axvline(0, color='black', linewidth=1)
ax.set_title('Improvement Compared With Baseline')
ax.set_xlabel('Score difference')
plt.tight_layout()
plt.show()

## 7.1 Per-Model Visual Diagnostics

These plots make it easier to see how each model behaves, not just which score is highest. The confusion matrices and prediction distributions use out-of-fold predictions, so every plotted prediction comes from a model that did not train on that row.

In [ ]:
model_names = [name for name in results_df.index if name != 'Baseline']
model_oof_predictions = {}

fig, axes = plt.subplots(len(model_names), 2, figsize=(11, 4 * len(model_names)))

for row, model_name in enumerate(model_names):
    model = best_estimators[model_name]

    # Out-of-fold predictions give a fairer diagnostic than training-set predictions.
    oof_pred = cross_val_predict(model, X, y, cv=cv, n_jobs=-1)
    model_oof_predictions[model_name] = oof_pred

    cm = confusion_matrix(y, oof_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[row, 0])
    axes[row, 0].set_title(f'{model_name}: Confusion Matrix')
    axes[row, 0].set_xlabel('Predicted')
    axes[row, 0].set_ylabel('Actual')

    pred_counts = pd.Series(oof_pred).value_counts().reindex([0, 1], fill_value=0)
    sns.barplot(x=pred_counts.index, y=pred_counts.values, ax=axes[row, 1], palette=['#4c78a8', '#f58518'])
    axes[row, 1].set_title(f'{model_name}: Predicted Class Distribution')
    axes[row, 1].set_xlabel('Predicted Survived')
    axes[row, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(len(model_names), 1, figsize=(10, 3 * len(model_names)), sharex=True)

if len(model_names) == 1:
    axes = [axes]

for ax, model_name in zip(axes, model_names):
    model = best_estimators[model_name]

    # Probability plots show model confidence, not only final 0/1 predictions.
    probabilities = cross_val_predict(model, X, y, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
    plot_data = pd.DataFrame({'Survival probability': probabilities, 'Actual Survived': y.map({0: 'No', 1: 'Yes'})})

    sns.histplot(
        data=plot_data,
        x='Survival probability',
        hue='Actual Survived',
        bins=20,
        kde=True,
        ax=ax
    )
    ax.axvline(0.5, color='black', linestyle='--', linewidth=1)
    ax.set_title(f'{model_name}: Out-of-Fold Survival Probabilities')
    ax.set_xlim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
best_model_name = results_df['F1'].idxmax()
final_model = best_estimators[best_model_name]

print(f'Best model: {best_model_name}')
print(results_df.loc[best_model_name].round(3))

## 8. Create Test Submission

This writes a sandbox submission file, so it does not overwrite the final assignment CSV.

In [ ]:
final_model.fit(X, y)
test_predictions = final_model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': test_predictions.astype(int),
})

submission_file = 'submission_sandbox.csv'
submission.to_csv(submission_file, index=False)

print(submission.shape)
print(submission['Survived'].value_counts().sort_index())
print(f'Saved {submission_file}')
submission.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

train['Survived'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color=['#4c78a8', '#f58518'])
axes[0].set_title('Training Target Distribution')
axes[0].set_xlabel('Survived')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

submission['Survived'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color=['#4c78a8', '#f58518'])
axes[1].set_title('Sandbox Prediction Distribution')
axes[1].set_xlabel('Predicted Survived')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## Quick Experiment Ideas

Try one change at a time:

- Set `age_imputation` to `'title_pclass'`.
- Turn `use_has_cabin` off.
- Turn `use_embarked_c` off.
- Add `max_depth` or `learning_rate` values to Gradient Boosting.
- Remove Voting Ensemble if you want a faster run.

After each change, compare `F1` and `Accuracy Δ vs Baseline`.